# Bias Analysis Notebook

This notebook analyzes fairness metrics.



In [4]:
import pandas as pd
import sklearn
from sklearn.utils import resample
df = pd.read_csv("fairvis_cluster.csv")
df.head()

,Age,Income,LoanAmount,CreditScore,MonthsEmployed,NumCreditLines,InterestRate,LoanTerm,DTIRatio,HasMortgage,...,LoanPurpose_home,LoanPurpose_other,InterestLevel_medium,InterestLevel_high,DTIBucket_medium,DTIBucket_high,DTIBucket_extreme,Default,prediction,cluster
0,31,33215,235022,771,67,3,10.53,36,0.90,1,...,0,0,1,0,0,0,0,1,1,47
1,23,37486,209844,498,12,2,4.40,12,0.61,1,...,0,0,0,0,0,0,0,1,1,37
2,60,45076,161301,603,62,1,8.70,12,0.52,1,...,1,0,0,0,0,0,0,0,0,18
3,63,33511,22824,745,62,4,15.15,36,0.56,0,...,0,0,1,0,0,0,0,0,0,6
4,60,21491,88388,542,3,3,16.78,48,0.54,0,...,0,0,1,0,0,0,0,0,1,27


In [6]:
# Global metrics
acc = (df['Default'] == df['prediction']).mean()
FN = ((df['Default']==1) & (df['prediction']==0)).sum()
TP = ((df['Default']==1) & (df['prediction']==1)).sum()
FNR = FN / (FN + TP + 1e-9)

FP = ((df['Default']==0) & (df['prediction']==1)).sum()
TN = ((df['Default']==0) & (df['prediction']==0)).sum()
FPR = FP / (FP + TN + 1e-9)

acc, FNR, FPR

(np.float64(0.7920277880821502),
 np.float64(0.2118609683527041),
 np.float64(0.20410359905817665))

In [8]:
feature = "CreditRisk"

group_metrics = df.groupby(feature).apply(
    lambda g: pd.Series({
        'size': len(g),
        'accuracy': (g['Default']==g['prediction']).mean(),
        'FNR': ((g['Default']==1) & (g['prediction']==0)).sum() / ((g['Default']==1).sum() + 1e-9),
        'FPR': ((g['Default']==0) & (g['prediction']==1)).sum() / ((g['Default']==0).sum() + 1e-9)
    })
)

group_metrics


C:\Users\natta\AppData\Local\Temp\ipykernel_61380\241817616.py:3: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  group_metrics = df.groupby(feature).apply(


,size,accuracy,FNR,FPR
CreditRisk,,,,
0,13021.0,0.795407,0.234456,0.177648
1,16632.0,0.789382,0.195657,0.226683


In [ ]:
categorical_features = [
    c for c in df.columns
    if df[c].nunique() <= 10 and c not in ['Default', 'prediction']
]

bias_table = []

for feat in categorical_features:
    groups = df.groupby(feat)
    for value, g in groups:
        bias_table.append({
            'feature': feat,
            'value': value,
            'size': len(g),
            'accuracy': (g['Default']==g['prediction']).mean(),
            'FNR': ((g['Default']==1) & (g['prediction']==0)).sum() / ((g['Default']==1).sum() + 1e-9),
            'FPR': ((g['Default']==0) & (g['prediction']==1)).sum() / ((g['Default']==0).sum() + 1e-9),
        })

bias_df = pd.DataFrame(bias_table)
bias_df.sort_values("FNR", ascending=False).head(20)
